# Whole-image as the PRIMARY route -- test-set pass (Phase 5)

Transcribes every one of the 3,300 test specimens from its **original uncropped
photo**, greedy (`temperature=0.0`, `n_samples=1`) at `MEDIA_RESOLUTION_HIGH` --
the config validated on the full 200-row dev set in
`docs/superpowers/plans/2026-08-21-transcription-quality-push.md` (D4): locality
mean NED 0.1655 -> 0.1224 with a 90% paired-bootstrap CI excluding 0, oracle-rank
AURC floor 0.0121 -> 0.0077, dev AURC 0.0487 -> 0.0353.

This is **not** the existing whole-image *fallback* pass
(`notebooks/kaggle_run_gemini_test_wholeimage.ipynb`), which only fires on the
1,041 specimens the per-card pass left both-fields-MISSING. This one replaces the
per-card route entirely; per-card predictions stay useful, but as an independent
*agreement* signal for confidence rather than as the text.

Runs one of the two API keys. A local invocation walks the same test list in
**reverse** with the other key, so the two meet in the middle with no
coordination round-trip; a small overlap where they meet is harmless (the merge
is a union keyed by `source_image_file`).

Needs a `GEMINI_API_KEY` secret (Add-ons -> Secrets), a *different* free-tier key
than the local one.

## Cell 1: locate the attached inputs

In [ ]:
import glob
import os


def _find_dataset_dir(slug):
    """Kaggle mounts attached datasets under /kaggle/input/<slug>/ almost always,
    but competition data sometimes nests under /kaggle/input/competitions/<slug>/
    instead -- glob for it rather than hardcoding one path."""
    candidates = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    for c in candidates:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError(f"dataset {slug!r} not found under /kaggle/input (checked: {candidates})")


COMPETITION_DIR = _find_dataset_dir("museumscat-specimen-collection-annotation-task")
CODE_DIR = _find_dataset_dir("museumscat-gemini-code")
try:
    PROGRESS_DIR = _find_dataset_dir("museumscat-gemini-progress")
except FileNotFoundError:
    PROGRESS_DIR = None  # first run, nothing to resume from
print("competition:", COMPETITION_DIR)
print("code       :", CODE_DIR)
print("progress   :", PROGRESS_DIR)

## Cell 2: the one dependency Kaggle's image lacks

In [ ]:
!pip install -q google-genai

## Cell 3: GEMINI_API_KEY from Kaggle Secrets

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
print("GEMINI_API_KEY set:", bool(os.environ.get("GEMINI_API_KEY")))

## Cell 4: unpack this project's code and wire up `data/`

`museumscat-gemini-code` uploads a single `src.zip`, and Kaggle flattens a
single-zip dataset -- so `CODE_DIR` holds the `.py` files directly, with no
`src/` subfolder to strip.

This pass reads only `data/raw/images/` and `data/raw/test.csv`: no card crops,
no manifest, no relevance flags, no per-card predictions. That is the whole point
of the route.

In [ ]:
import shutil

REPO_DIR = "/kaggle/working/repo"
os.makedirs(os.path.join(REPO_DIR, "src"), exist_ok=True)
for py_file in glob.glob(os.path.join(CODE_DIR, "*.py")):
    shutil.copyfile(py_file, os.path.join(REPO_DIR, "src", os.path.basename(py_file)))
os.chdir(REPO_DIR)

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed/gemini", exist_ok=True)
shutil.copyfile(os.path.join(COMPETITION_DIR, "test.csv"), "data/raw/test.csv")
if not os.path.exists("data/raw/images"):
    os.symlink(os.path.join(COMPETITION_DIR, "images"), "data/raw/images")

OUTPUT_PATH = "data/processed/gemini/test_wholeimage_primary_kaggle.csv"
# Seed from the progress dataset so a second session resumes instead of redoing
# work -- Kaggle wipes /kaggle/working between runs, continuity is manual.
if PROGRESS_DIR:
    seed = os.path.join(PROGRESS_DIR, os.path.basename(OUTPUT_PATH))
    if os.path.isfile(seed):
        shutil.copyfile(seed, OUTPUT_PATH)
        print("seeded from progress dataset")

print(os.getcwd(), "| src files:", len(glob.glob("src/*.py")))
print("images:", len(os.listdir("data/raw/images")))

## Cell 5: the runner

Deliberately inlined rather than added to `src/`: it imports the prompt, response
schema, parser, rate limiter, client builder and model rotation from the code
dataset, so it differs from production only in the `GenerateContentConfig` and in
reading the whole photo instead of a crop. Nothing here can drift from `src/`
except the config under test.

Resumable by append + skip, flushed per row, and it **stops** on a daily-quota
429 rather than retrying it (a retried daily 429 cost ~1.5h of a Kaggle session
on 2026-08-21).

In [ ]:
import csv
import json
import time

import httpx
import pandas as pd
from google.genai import errors as genai_errors
from google.genai import types

from src.gemini_batch_runner import (
    ModelRotation, OUTPUT_COLUMNS_WHOLEIMAGE, already_done, is_daily_quota_exhausted)
from src.gemini_transcribe import (
    RESPONSE_SCHEMA, RateLimiter, WHOLE_IMAGE_PROMPT, _parse_response_json, build_client)

MEDIA_RESOLUTION = "MEDIA_RESOLUTION_HIGH"
MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
RPM = 12


def call_once(client, image_bytes, model, rate_limiter):
    rate_limiter.wait()
    response = client.models.generate_content(
        model=model,
        contents=[{"inline_data": {"data": image_bytes, "mimeType": "image/jpeg"}}],
        config=types.GenerateContentConfig(
            system_instruction=WHOLE_IMAGE_PROMPT,
            temperature=0.0,
            response_mime_type="application/json",
            response_json_schema=RESPONSE_SCHEMA,
            media_resolution=MEDIA_RESOLUTION,
        ),
    )
    return _parse_response_json(response.text)


def run(output_path, limit=None):
    rotation = ModelRotation(MODELS)
    image_files = pd.read_csv("data/raw/test.csv")["image_file"].tolist()
    done = already_done(output_path, key_column="source_image_file")
    pending = [f for f in image_files if f not in done]
    if limit is not None:
        pending = pending[:limit]
    print(f"[setup] {len(image_files)} test specimens, {len(done)} done, "
          f"{len(pending)} this run (media_resolution={MEDIA_RESOLUTION})", flush=True)
    if not pending:
        print("[done] nothing to do", flush=True)
        return

    client = build_client(os.environ["GEMINI_API_KEY"])
    rate_limiter = RateLimiter(max_calls_per_minute=RPM)
    write_header = not os.path.exists(output_path)
    started = time.time()
    with open(output_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_COLUMNS_WHOLEIMAGE)
        if write_header:
            writer.writeheader()
        for i, image_file in enumerate(pending, start=1):
            with open(os.path.join("data/raw/images", image_file), "rb") as img_f:
                image_bytes = img_f.read()
            sample = None
            while sample is None:
                try:
                    sample = call_once(client, image_bytes, rotation.current, rate_limiter)
                    sample["_model"] = rotation.current
                except genai_errors.ClientError as e:
                    if e.code != 429:
                        raise
                    if is_daily_quota_exhausted(e):
                        if not rotation.advance():
                            print(f"[stop] every model exhausted after {i - 1}/{len(pending)} "
                                  "-- rerun later to resume", flush=True)
                            return
                        continue
                    print("!! per-minute 429, backing off 60s", flush=True)
                    time.sleep(60)
                except (genai_errors.ServerError, httpx.TimeoutException, httpx.ConnectError) as e:
                    print(f"!! {image_file}: {type(e).__name__}, retrying in 5s", flush=True)
                    time.sleep(5)
            writer.writerow({
                "source_image_file": image_file,
                "verbatimDate": sample["verbatimDate"],
                "verbatimDate_confidence": 1.0,
                "verbatimLocality": sample["verbatimLocality"],
                "verbatimLocality_confidence": 1.0,
                "models_used": rotation.current,
                "raw_samples_json": json.dumps([sample]),
            })
            f.flush()
            if i % 25 == 0 or i == len(pending):
                print(f"[{i}/{len(pending)}] {(time.time() - started) / 60:.1f} min", flush=True)
    print("[done]", flush=True)

## Cell 6: smoke test -- 3 requests, then look at them

Spends 3 requests to prove the wiring before committing a session to it.

In [ ]:
SMOKE_PATH = "/kaggle/working/_smoke.csv"
run(SMOKE_PATH, limit=3)
print()
print(pd.read_csv(SMOKE_PATH)[["source_image_file", "verbatimDate", "verbatimLocality"]].to_string())

## Cell 7: the real run

Walks the test list **forward**; the local key walks it in reverse. Stops cleanly
when both models' daily quota is gone -- just let it run and see how far it gets.
Kaggle caps a session at 12h and each request takes ~22s (a 45 MP upload), so one
session covers at most ~1,900 specimens even with quota to spare.

In [ ]:
run(OUTPUT_PATH)

## Cell 8: persist progress

`/kaggle/working` is wiped between sessions -- **Save & Run All** so this output
survives as the notebook's own output. To resume later, download it, merge with
whatever the local run produced (union on `source_image_file`), and push the
merged file as a new version of `museumscat-gemini-progress`:

```
kaggle datasets version -p data/processed/gemini/kaggle_progress_dataset -m "..."
```

Skipping that merge is how a Kaggle run's contribution gets silently clobbered by
a stale re-upload -- it has nearly happened once already.

In [ ]:
n = sum(1 for _ in open(OUTPUT_PATH)) - 1 if os.path.exists(OUTPUT_PATH) else 0
print(f"{n} rows at {os.path.join(REPO_DIR, OUTPUT_PATH)} -- visible in the Output pane after Save Version")